# Ingest drivers.json file Assignment


## Assignment
- create a DF loading the drivers.json file in the raw container
- rename columns to (driver_id,driver_ref)
- add a new column name from (name,firebanem name.surname)
- add a new columns ingestion_timestampt
- drop the columns (name,firebanem name.surname)
- save the file in parquet file in the processed container.
- verify the schema of the partquet file

In [0]:
%run "../includes/common_functions"

In [0]:
%run "../includes/configuration"

In [0]:
dbutils.widgets.text("p_date_source","")
#dbutils.widgets.dropdown("p_date_source","Testing",["Testing","Production"])
v_data_source=dbutils.widgets.get("p_date_source")

In [0]:
from pyspark.sql.functions import current_timestamp,col,concat,lit
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType


In [0]:
name_schema=StructType(fields=[StructField("forename",StringType(),True),
                                 StructField("surname",StringType(),True)])
drivers_schema=StructType(fields=[StructField("driverId",IntegerType(),True),
                                 StructField("driverRef",StringType(),True),
                                 StructField("number",IntegerType(),True),
                                 StructField("code",StringType(),True),
                                 StructField("name",name_schema),
                                 StructField("dob",StringType(),True),
                                 StructField("nationality",StringType(),True)])

In [0]:
drivers_df = spark.read.schema(drivers_schema).json(f"{raw_folder_path}/drivers.json")


In [0]:
drivers_df.printSchema()

In [0]:

drivers_df = drivers_df.withColumn("name",concat(col("name.forename"),lit(" "),col("name.surname"))).withColumnRenamed("driverId","driver_id").withColumnRenamed("driverRef","driver_ref")

In [0]:
drivers_df= add_ingestion_timestamp(drivers_df)
drivers_df = add_data_source(drivers_df,v_data_source)

In [0]:
#drivers_df=drivers_df.select(col("driverId").alias("driver_id"),col("driverRef").alias("driver_ref"),col("number"),col("code"),col("name"),col("dob"),col("nationality"),col("ingestion_timestamp"))

In [0]:
display(drivers_df)

## Write DF into parquet file

In [0]:
#drivers_df.write.mode("overwrite").parquet(f"{processed_folder_path}/drivers")

In [0]:
drivers_df.write.mode("overwrite").format("parquet").saveAsTable("f1_processed_db.drivers")

In [0]:
#df=spark.read.parquet(f"{processed_folder_path}/drivers")
#df.printSchema()

In [0]:
dbutils.notebook.exit("Success")